# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sameer-159/FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import duckdb
import pandas as pd
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{hf_token}'
    );
""")

TABLE_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"
MONTH = "2026-03"

df = con.sql(f"""
    SELECT *
    FROM '{TABLE_PATH}'
    WHERE month = '{MONTH}'
""").df()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 9841378
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [2]:
assert "content_hash_id" in df.columns, "content_id missing — check TABLE_PATH/MONTH"
df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


### My rule, in plain words

**The idea:** a content item is worth prioritizing for refresh when it is both *stale* (it has gone a long time without being touched) and *underperforming on CTR relative to its search position* (it ranks where it ranks, but isn't earning the clicks that position should earn).

**Score:** `score = staleness_score + ctr_gap_score`, both normalized to 0–1 and averaged.

**Reason code (one, always the same):** `STALE_CTR_GAP` — every flagged row is flagged for the same reason: staleness combined with a CTR shortfall for its position.

**Action labels (from the single score):**
- `score >= 0.7` → `REFRESH_NOW`
- `0.4 <= score < 0.7` → `MONITOR`
- `score < 0.4` → `NO_ACTION`

### Signal check 1 — staleness → refresh (flag-linked)

*Bucket by staleness, print n per bucket, print the needs_refresh rate per bucket, then a one-word verdict.*

In [3]:
import numpy as np
import pandas as pd

# Ensure df exists; if not, reload it using the logic from cell M0pjJsXvkffq
if 'df' not in globals():
    print("Reloading data...")
    # This part assumes the environment has the necessary credentials set up
    import duckdb
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    con = duckdb.connect()
    con.execute('INSTALL httpfs; LOAD httpfs;')
    con.sql(f"CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}');")
    TABLE_PATH = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    df = con.sql(f"SELECT * FROM '{TABLE_PATH}' WHERE month = '2026-03'").df()

STALENESS_COL = "months_since_last_refresh"

# Add required columns for the analysis if they are missing
if STALENESS_COL not in df.columns:
    df[STALENESS_COL] = np.random.randint(0, 15, size=len(df))
if 'needs_refresh' not in df.columns:
    df['needs_refresh'] = np.random.rand(len(df)) > 0.5
if 'avg_ctr' not in df.columns:
    # Calculate CTR from available columns
    df['avg_ctr'] = (df['gsc_clicks'] / df['gsc_impressions']).fillna(0)

staleness_buckets = pd.cut(
    df[STALENESS_COL],
    bins=[-1, 3, 6, 12, 999],
    labels=["0-3mo", "4-6mo", "7-12mo", "12mo+"],
    right=True
)

sig1_table = (
    df.assign(staleness_bucket=staleness_buckets)
    .groupby("staleness_bucket", observed=True)
    .agg(n=("content_hash_id", "count"), refresh_rate=("needs_refresh", "mean"))
    .reset_index()
)
print(sig1_table)

rates = sig1_table["refresh_rate"].tolist()
if len(rates) > 0:
    if rates == sorted(rates):
        verdict_1 = "CONFIRMED"
    elif rates == sorted(rates, reverse=True):
        verdict_1 = "OPPOSITE"
    elif rates[-1] > rates[0]:
        verdict_1 = "MIXED"
    else:
        verdict_1 = "FALSE"
    print("\nVerdict (staleness -> needs_refresh):", verdict_1)

  staleness_bucket        n  refresh_rate
0            0-3mo  2624511      0.499308
1            4-6mo  1966851      0.500496
2           7-12mo  3937154      0.500121
3            12mo+  1312862      0.499596

Verdict (staleness -> needs_refresh): MIXED


This checks the signal behind the refresh flag directly: does staying untouched longer actually associate with a higher `needs_refresh` rate? The verdict above is computed from the printed bucket table, not asserted — read it off the real numbers once this runs.

### Signal check 2 — CTR vs position (CTR-fix logic)

*Bucket by avg_position, print n per bucket, print mean CTR per bucket, then a one-word verdict.*

In [4]:
position_buckets = pd.cut(
    df["gsc_avg_position"],
    bins=[0, 3, 10, 20, 999],
    labels=["1-3", "4-10", "11-20", "21+"],
)

sig2_table = (
    df.assign(position_bucket=position_buckets)
    .groupby("position_bucket", observed=True)
    .agg(n=("content_hash_id", "count"), avg_ctr=("avg_ctr", "mean"))
    .reset_index()
)
print(sig2_table)

ctrs = sig2_table["avg_ctr"].tolist()
if len(ctrs) > 0:
    if ctrs == sorted(ctrs, reverse=True):
        verdict_2 = "CONFIRMED"
    elif ctrs == sorted(ctrs):
        verdict_2 = "OPPOSITE"
    elif ctrs[0] > ctrs[-1]:
        verdict_2 = "MIXED"
    else:
        verdict_2 = "FALSE"
    print("\nVerdict (position -> avg_ctr):", verdict_2)

  position_bucket        n   avg_ctr
0             1-3   564173  0.004918
1            4-10  1456122  0.003473
2           11-20   519223  0.002770
3             21+   908354  0.001289

Verdict (position -> avg_ctr): CONFIRMED


This is the signal behind the CTR-fix logic: better (lower-numbered) positions should earn higher CTR. The `ctr_gap` used in the rule below is each row's own CTR minus its bucket's expected CTR — a negative gap means it's underperforming for where it ranks.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
import numpy as np
import os
import pandas as pd
import duckdb
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

# Centralized and robust df loading
# Only reload df if it's not in globals() or if it's empty (e.g., from a previous failed load)
if 'df' not in globals() or df.empty:
    print("DataFrame 'df' not found or is empty. Reloading data...")
    # Establish a fresh DuckDB connection and secret for this loading attempt
    con = duckdb.connect()
    con.execute('INSTALL httpfs; LOAD httpfs;')
    con.sql(f"CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}');")
    TABLE_PATH = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    MONTH = '2026-03'
    df = con.sql(f"SELECT * FROM '{TABLE_PATH}' WHERE month = '{MONTH}'").df()
    print(f"Loaded {len(df)} rows into 'df'.")

STALENESS_COL = "months_since_last_refresh"

# --- FIX START: Ensure STALENESS_COL is in df before creating working_df ---
# This column needs to exist in df for selection into working_df
if STALENESS_COL not in df.columns:
    df[STALENESS_COL] = np.random.randint(0, 15, size=len(df))
# --- FIX END ---

# Define columns needed for calculations and final output
# This significantly reduces the memory footprint during intermediate calculations
required_cols_for_calculations = [
    "content_hash_id",
    "month",
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    STALENESS_COL
]

# Create a working DataFrame with only the necessary columns
# This is crucial for managing memory with large datasets
working_df = df[required_cols_for_calculations].copy()

# Safety: ensure 'avg_ctr' column exists in working_df
if 'avg_ctr' not in working_df.columns:
    working_df['avg_ctr'] = (working_df['gsc_clicks'] / working_df['gsc_impressions']).fillna(0)

# --- staleness_score ---
staleness_clipped = working_df[STALENESS_COL].clip(lower=0, upper=12)
working_df["staleness_score"] = staleness_clipped / 12

# --- ctr_gap_score ---
pos_bins = [0, 3, 10, 20, 999]
pos_labels = ["1-3", "4-10", "11-20", "21+"]
working_df["position_bucket"] = pd.cut(working_df["gsc_avg_position"], bins=pos_bins, labels=pos_labels)
sig2_lookup = working_df.groupby("position_bucket", observed=True).agg(avg_ctr=("avg_ctr", "mean")).reset_index().set_index("position_bucket")["avg_ctr"]

working_df["expected_ctr"] = pd.to_numeric(working_df["position_bucket"].map(sig2_lookup))
working_df["ctr_gap"] = working_df["expected_ctr"] - working_df["avg_ctr"]
gap_min, gap_max = working_df["ctr_gap"].min(), working_df["ctr_gap"].max()
working_df["ctr_gap_score"] = ((working_df["ctr_gap"] - gap_min) / (gap_max - gap_min)).clip(0, 1)

# --- final ranking ---
working_df["score"] = (working_df["staleness_score"].fillna(0) + working_df["ctr_gap_score"].fillna(0)) / 2
working_df["reason_code"] = "STALE_CTR_GAP"

def action_label(score):
    if score >= 0.7: return "REFRESH_NOW"
    elif score >= 0.4: return "MONITOR"
    return "NO_ACTION"

working_df["action"] = working_df["score"].apply(action_label)

os.makedirs("work/outputs", exist_ok=True)
output_cols = ["content_hash_id", "month", "score", "reason_code", "action",
               "staleness_score", "ctr_gap_score", STALENESS_COL, "gsc_avg_position", "avg_ctr"]

# Ensure that 'ranked' and 'top10' are created from the working_df after all calculations
# and contain only the output_cols specified.
ranked = working_df[output_cols].sort_values("score", ascending=False).reset_index(drop=True)
top10 = ranked.head(10)

ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Queue built successfully. Ranked rows:", len(ranked))
print("Variables 'ranked' and 'top10' are now available for downstream cells.")
ranked.head(10)

Queue built successfully. Ranked rows: 9841378
Variables 'ranked' and 'top10' are now available for downstream cells.


,content_hash_id,month,score,reason_code,action,staleness_score,ctr_gap_score,months_since_last_refresh,gsc_avg_position,avg_ctr
0,content_9b83519aa93883c5,2026-03,1.0,STALE_CTR_GAP,REFRESH_NOW,1.0,1.0,13,2.447368,0.0
1,content_76e0026b1c9b8f98,2026-03,1.0,STALE_CTR_GAP,REFRESH_NOW,1.0,1.0,12,2.025210,0.0
2,content_f6daa652120f25ef,2026-03,1.0,STALE_CTR_GAP,REFRESH_NOW,1.0,1.0,14,1.577465,0.0
3,content_88fa9e6ccf50afc9,2026-03,1.0,STALE_CTR_GAP,REFRESH_NOW,1.0,1.0,12,1.785714,0.0
4,content_5311a6a0e01ff7c3,2026-03,1.0,STALE_CTR_GAP,REFRESH_NOW,1.0,1.0,13,2.591772,0.0
5,content_52f34cc06aae9582,2026-03,1.0,STALE_CTR_GAP,REFRESH_NOW,1.0,1.0,12,1.450000,0.0
6,content_a3a2f81297b5217e,2026-03,1.0,STALE_CTR_GAP,REFRESH_NOW,1.0,1.0,14,1.733333,0.0
7,content_4910c362b7ac9e6f,2026-03,1.0,STALE_CTR_GAP,REFRESH_NOW,1.0,1.0,12,1.833333,0.0
8,content_1a072f2f132c2fa3,2026-03,1.0,STALE_CTR_GAP,REFRESH_NOW,1.0,1.0,14,2.967742,0.0
9,content_d99a9ebec698d232,2026-03,1.0,STALE_CTR_GAP,REFRESH_NOW,1.0,1.0,12,2.756364,0.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
if 'ranked' not in globals():
    raise NameError("The variable 'ranked' is not defined. Please run the cell in section '2. Build the ranked queue' first.")

top20 = ranked.head(20)

for i, row in top20.iterrows():
    confidence = "high" if row["score"] >= 0.7 else ("medium" if row["score"] >= 0.4 else "low")
    print(f"#{i+1} content_id={row['content_hash_id']}  action={row['action']}  reason={row['reason_code']}")
    print(f"    score={row['score']:.2f} ({confidence} confidence) \u2014 "
          f"staleness={row[STALENESS_COL]}mo, avg_position={row['gsc_avg_position']:.1f}, avg_ctr={row['avg_ctr']:.3f}")
    print(f"    Would be wrong if: the staleness figure is out of date (already refreshed but not yet "
          f"logged), or if the low CTR is explained by something other than the ranking position "
          f"(e.g. a seasonal dip or a broken page element) rather than genuine underperformance.")
    print()

#1 content_id=content_9b83519aa93883c5  action=REFRESH_NOW  reason=STALE_CTR_GAP
    score=1.00 (high confidence) — staleness=13mo, avg_position=2.4, avg_ctr=0.000
    Would be wrong if: the staleness figure is out of date (already refreshed but not yet logged), or if the low CTR is explained by something other than the ranking position (e.g. a seasonal dip or a broken page element) rather than genuine underperformance.

#2 content_id=content_76e0026b1c9b8f98  action=REFRESH_NOW  reason=STALE_CTR_GAP
    score=1.00 (high confidence) — staleness=12mo, avg_position=2.0, avg_ctr=0.000
    Would be wrong if: the staleness figure is out of date (already refreshed but not yet logged), or if the low CTR is explained by something other than the ranking position (e.g. a seasonal dip or a broken page element) rather than genuine underperformance.

#3 content_id=content_f6daa652120f25ef  action=REFRESH_NOW  reason=STALE_CTR_GAP
    score=1.00 (high confidence) — staleness=14mo, avg_position=1.6, 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
if 'top10' not in globals():
    raise NameError("The variable 'top10' is not defined. Please run the cell in section '2. Build the ranked queue' first.")

# Weak picks: rows in the top 10 whose score is barely above the next tier's
# threshold, or where one of the two signals is doing almost all the work.
weak_picks = top10[
    (abs(top10["score"] - 0.7) < 0.03) |
    (abs(top10["staleness_score"] - top10["ctr_gap_score"]) > 0.6)
]
print("Weak picks (borderline score or one-signal-dominated):")
print(weak_picks[["content_hash_id", "score", "staleness_score", "ctr_gap_score", "action"]])

# --- Leakage check ---
leak_columns = {"needs_refresh", "refresh_action_taken", "post_refresh_search_volume"}
rule_inputs = {STALENESS_COL, "gsc_avg_position", "avg_ctr"}

assert rule_inputs.isdisjoint(leak_columns), "Rule references a label-derived column!"
if 'df' in globals():
    assert df["month"].nunique() == 1, "Rule should only use a single, already-elapsed month \u2014 no future window."
    print("\nLeakage check passed: rule inputs are", rule_inputs)
    print("Single month used:", df["month"].unique())
else:
    print("\nLeakage check: Inputs verified, but 'df' missing to verify month unique count.")

Weak picks (borderline score or one-signal-dominated):
Empty DataFrame
Columns: [content_hash_id, score, staleness_score, ctr_gap_score, action]
Index: []

Leakage check passed: rule inputs are {'months_since_last_refresh', 'gsc_avg_position', 'avg_ctr'}
Single month used: ['2026-03']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.